# 01 - Data Cleaning

In [ ]:
# Import Libraries for Data Cleaning 
import numpy as np 
import pandas as pd


# Reading each file into its own data frame
monday = pd.read_csv("../data/raw/Monday-WorkingHours.pcap_ISCX.csv")
tuesday = pd.read_csv("../data/raw/Tuesday-WorkingHours.pcap_ISCX.csv")
wednesday = pd.read_csv("../data/raw/Wednesday-workingHours.pcap_ISCX.csv")
thursday_morning = pd.read_csv("../data/raw/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv")
thursday_afternoon = pd.read_csv("../data/raw/Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv")
friday_morning = pd.read_csv("../data/raw/Friday-WorkingHours-Morning.pcap_ISCX.csv")
friday_portscan = pd.read_csv("../data/raw/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv")
friday_ddos = pd.read_csv("../data/raw/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv")

# Tag each data frame with its file so rows can be traced back
monday["source_file"] = "Monday"
tuesday["source_file"] = "Tuesday"
wednesday["source_file"] = "Wednesday"
thursday_morning["source_file"] = "Thursday-Morning-WebAttacks"
thursday_afternoon["source_file"] = "Thursday-Afternoon-Infiltration"
friday_morning["source_file"] = "Friday-Morning"
friday_portscan["source_file"] = "Friday-Afternoon-PortScan"
friday_ddos["source_file"] = "Friday-Afternoon-DDoS"

# Stack all eight data frames, numbering rows from 0
dataDf = pd.concat([monday, tuesday, wednesday, thursday_morning,
                    thursday_afternoon, friday_morning, friday_portscan, friday_ddos], ignore_index=True)

# Remove spaces from the column names
dataDf.columns = dataDf.columns.str.strip()


print(f"Total Rows: {len(dataDf)}")

Total Rows: 2830743


In [35]:
# Should print (2830743, 80): 79 original columns plus source_file
print(dataDf.shape)

# Should list all eight sources with their row counts
print(dataDf["source_file"].value_counts())

(2830743, 80)
source_file
Wednesday                          692703
Monday                             529918
Tuesday                            445909
Thursday-Afternoon-Infiltration    288602
Friday-Afternoon-PortScan          286467
Friday-Afternoon-DDoS              225745
Friday-Morning                     191033
Thursday-Morning-WebAttacks        170366
Name: count, dtype: int64


## Step 1: Missing values

### Quantify Missing Values

In [36]:
# Count missing (NaN) values in every column
missing_counts = dataDf.isna().sum()

# Show only the columns that have at least one missing value
missing_counts[missing_counts > 0]

Flow Bytes/s    1358
dtype: int64

In [37]:
# Replace positive and negative infinity with NaN so they are counted as missing
cleanDf = dataDf.replace([np.inf, -np.inf], np.nan)

# Count missing (NaN) values in every column
missing_counts = cleanDf.isna().sum()

# Show only columns that have at least one missing value
missing_counts[missing_counts > 0]

Flow Bytes/s      2867
Flow Packets/s    2867
dtype: int64

**Result:** Only two columns have problem values.
- Flow Bytes/s: 1,358 missing + 1,509 infinite = 2,867
- Flow Packets/s: 0 missing + 2,867 infinite = 2,867

This is 0.10% of the 2,830,743 rows. `isna()` does not count infinity as missing,
so infinite values were replaced with NaN first using `replace()`.

### Investigate the pattern
Look at the rows with missing values to find out what they have in common.

In [38]:
# Keeping only rows that have at least one missing value
missing_rows = cleanDf[cleanDf.isna().any(axis=1)]

# Counting those rows
print(f"Rows with a missing value: {len(missing_rows)}" )

Rows with a missing value: 2867


Both of our columns `Flow Bytes/s` & `Flow Packets/s`, have 2,867 missing values, but only 2,867 rows have a missing value in total. Having only 2,867 missing values means that ultimately both columns have the same missing rows. We will further investigate this by finding the Flow Duration values of these rows, since both columns are rates calculated by dividing by the flow duration. 

In [39]:
# Finding the flow duration values in the rows with the missing values
print(missing_rows['Flow Duration'].value_counts())

# Set flow_durationZeroes by checking whole dataset to find zero values
flow_durationZeroes = (cleanDf['Flow Duration'] == 0).sum()

# Count how many rows in the entire dataset have a zero flow duration
print(f"Rows with Flow Duration = 0: {flow_durationZeroes}")


Flow Duration
0    2867
Name: count, dtype: int64
Rows with Flow Duration = 0: 2867


**Result**: All 2,867 rows with missing values have a Flow Duration of zero, and these are the only rows in the dataset with a Flow Duration of 0. Since Flow Bytes/s and
Flow Packets/s are rates calculated by dividing by the flow duration, a duration of 0
causes a division by zero, which produces the missing (NaN) and infinite values.

In [40]:
# Which days the missing rows came from
print(missing_rows['source_file'].value_counts())


# Counting missing rows per label
affected = missing_rows['Label'].value_counts()

# Counting all rows per label
total = cleanDf['Label'].value_counts()

# Percentage of each label that is affected
share = affected/ total * 100

# Put the three columns side by side
summary = pd.DataFrame({"affected": affected, "total": total, "share_%": share.round(2)})

# Display the three columns side by side (Labels with no missing rows show 0)
display(summary.fillna(0))

source_file
Wednesday                          1297
Monday                              437
Friday-Afternoon-PortScan           371
Tuesday                             264
Thursday-Afternoon-Infiltration     207
Thursday-Morning-WebAttacks         135
Friday-Morning                      122
Friday-Afternoon-DDoS                34
Name: count, dtype: int64


,affected,total,share_%
Label,,,
BENIGN,1777.0,2273097,0.08
Bot,10.0,1966,0.51
DDoS,2.0,128027,0.00
DoS GoldenEye,0.0,10293,0.00
DoS Hulk,949.0,231073,0.41
DoS Slowhttptest,0.0,5499,0.00
DoS slowloris,0.0,5796,0.00
FTP-Patator,3.0,7938,0.04
Heartbleed,0.0,11,0.00


### Decision: Remove the Rows

We remove the 2,867 rows with missing values instead of filling them in or keeping them. Filling them in does not make sense, because `Flow Bytes/s` and `Flow Packets/s` are per second rates, and a flow that lasted zero seconds wouldn't provide a meaningful rate, so any value we filled in would be made up. Keeping them is also not an option, because NaN and infinite values would make calculations such as averages not work. This leaves removing them as our best option, since the missing values affect 0.10% of the data and at most 0.51% of any single traffic class (Bot), and the rarest attack classes are not affected.

In [41]:
# Count rows before removing
rows_before = len(cleanDf)

# Remove every row that has a missing value
cleanDf = cleanDf.dropna()

# Count rows after removing
rows_after = len(cleanDf)

# Confirmation that rows were removed.
print(f"Rows Before Removing: {rows_before}")
print(f"Rows After Removing: {rows_after}")
print(f"Rows Removed: {rows_before - rows_after}")

Rows Before Removing: 2830743
Rows After Removing: 2827876
Rows Removed: 2867


**Result**: 2,867 rows were removed from the dataset. Our data went from 2,830,743 to 2,827,876 rows and no missing or infinite values are left.

## Step 2: Duplicates and Inconsistencies

### Finding Duplicate rows

In [42]:
# Checking for duplicates while ignoring the source_file column
dup_rows = cleanDf.drop(columns="source_file").duplicated()

# Number of duplicated rows
dup_rows_num = dup_rows.sum()

# Percentage of duplicated rows in the total dataset
percent_dup_rows = round(dup_rows_num/ len(cleanDf) * 100, 2)

# Count the duplicates and their percentage
print(f"Duplicate rows: {dup_rows_num}")
print(f"Percent of data that is duplicated: {percent_dup_rows} %")

Duplicate rows: 307078
Percent of data that is duplicated: 10.86 %


**Result:** 307,078 rows (10.86%) are exact duplicates of other rows.

### Which Classes have duplicates ? 

In [43]:
# Counting duplicate rows per label
dup_per_label = cleanDf[dup_rows]['Label'].value_counts()

# Counting all rows per label
total = cleanDf['Label'].value_counts()

# Percentage of each label that is duplicated 
share = dup_per_label / total * 100

# Putting the three columns side by side
summary = pd.DataFrame({"duplicates": dup_per_label, 
                        "total": total, 
                        "share_%": share.round(2)})

# Display the table (labels with no duplicates show 0)
display(summary.fillna(0))

,duplicates,total,share_%
Label,,,
BENIGN,176263.0,2271320,7.76
Bot,8.0,1956,0.41
DDoS,11.0,128025,0.01
DoS GoldenEye,7.0,10293,0.07
DoS Hulk,57278.0,230124,24.89
DoS Slowhttptest,271.0,5499,4.93
DoS slowloris,411.0,5796,7.09
FTP-Patator,2004.0,7935,25.26
Heartbleed,0.0,11,0.00


**Result**: The duplicates are not distributed evenly across the classes. Approximately 45% of SSH-Patator and 43% of PortScan flows are duplicates, compared to 8% of BENIGN flows. Repetitive attacks such as port scans and password guessing (SSH-Patator) send the same requests many times, which creates flows having identical statistics.

### Decision: Remove the Duplicate Rows

We remove the duplicate rows and keep one copy of each. Without IP addresses or timestamps, we're not able to tell whether the identical rows are separate flows that look the same or the same flow recorded more than once, and Lanvin et al. (2023) documented packet duplication in CICIDS2017. If duplicates were kept, identical rows could end up in both of our training and testing data, which could make results look better than they really are. This matters especially for our fourth research question, which trains on one day and tests on another. Although removing duplicates reduces some attack classes (approximately 45% of SSH-Patator and 43% of PortScan), every class still keeps enough rows to analyze, and the rarest classes are not affected. 

In [44]:
# Count rows before removing
rows_before = len(cleanDf)

# Remove duplicate rows (ignoring source_file), keeping the first copy of each
cleanDf = cleanDf.drop_duplicates(subset=cleanDf.columns.drop("source_file"))

# Count rows after removing
rows_after = len(cleanDf)

print(f"Rows Before Removing: {rows_before}")
print(f"Rows After Removing: {rows_after}")
print(f"Rows Removed: {rows_before - rows_after}")

Rows Before Removing: 2827876
Rows After Removing: 2520798
Rows Removed: 307078


In [45]:
# Rows left in each class after removing duplicates
display(cleanDf['Label'].value_counts())

Label
BENIGN                        2095057
DoS Hulk                       172846
DDoS                           128014
PortScan                        90694
DoS GoldenEye                   10286
FTP-Patator                      5931
DoS slowloris                    5385
DoS Slowhttptest                 5228
SSH-Patator                      3219
Bot                              1948
Web Attack � Brute Force         1470
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64

**Result:** 307,078 duplicated rows were removed from the dataset. Our data went from 2,827,876 rows to 2,520,798 rows but all the traffic classes still have rows. 

### Fixing Inconsistencies

The three Web Attack Labels contain a broken character "�" instead of a dash. An example of this includes, "Web Attack � Brute Force". This character gets replaced with a normal dash so the labels can be displayed correctly.

In [46]:
# Web attack labels before 
print("Before:")
for label in cleanDf['Label'].unique():
    if 'Web Attack' in label:
        print(label)

# Replace the broken character with a dash
cleanDf['Label'] = cleanDf['Label'].str.replace('�', '-')

# Web attack labels after
print("\nAfter:")
for label in cleanDf['Label'].unique():
    if 'Web Attack' in label:
        print(label)


Before:
Web Attack � Brute Force
Web Attack � XSS
Web Attack � Sql Injection

After:
Web Attack - Brute Force
Web Attack - XSS
Web Attack - Sql Injection


In [47]:
# Showing all label names with changes to Web Attack
print("Labels:")
for label in cleanDf["Label"].unique():
    print(label)

Labels:
BENIGN
FTP-Patator
SSH-Patator
DoS slowloris
DoS Slowhttptest
DoS Hulk
DoS GoldenEye
Heartbleed
Web Attack - Brute Force
Web Attack - XSS
Web Attack - Sql Injection
Infiltration
Bot
PortScan
DDoS


**Result:** The three Web Attack labels now have a dash instead of the broken character "�". 

### Fixing inconsistent column names

In the raw files, 65 of the 79 column names start with an extra space. These extra spaces were removed when loading the data with `.str.strip()`. 

### Dropping Columns with no insights

We drop 9 columns, eight of these columns always contain 0 in every row (Bwd PSH Flags, Bwd URG Flags, and
the six bulk features), so they cannot help separate traffic types. The other column, Fwd Header Length.1, is a duplicate of Fwd Header Length that appears twice in the CSV files and is not defined in the CICFlowMeter documentation. The check below confirms both.


A column that never changes cannot help separate traffic types, and it can cause issues later, since methods such as standardization divide by the standard deviation, which is 0 for these columns. Also having a repeated column name, only repeats information.

In [48]:
# Columns that are always 0
zero_columns = [
    "Bwd PSH Flags",
    "Bwd URG Flags",
    "Fwd Avg Bytes/Bulk",
    "Fwd Avg Packets/Bulk",
    "Fwd Avg Bulk Rate",
    "Bwd Avg Bytes/Bulk",
    "Bwd Avg Packets/Bulk",
    "Bwd Avg Bulk Rate",
]

# Check to see if every value in each column is equal to zero
print("Always-zero check:")
for col in zero_columns:
    print(col, (cleanDf[col] == 0).all())


# Check that Fwd Header Length.1 is identical to Fwd Header Length
print("Fwd Header Length.1 = Fwd Header Length:", cleanDf["Fwd Header Length.1"].equals(cleanDf["Fwd Header Length"]))    

Always-zero check:
Bwd PSH Flags True
Bwd URG Flags True
Fwd Avg Bytes/Bulk True
Fwd Avg Packets/Bulk True
Fwd Avg Bulk Rate True
Bwd Avg Bytes/Bulk True
Bwd Avg Packets/Bulk True
Bwd Avg Bulk Rate True
Fwd Header Length.1 = Fwd Header Length: True


In [49]:
# Setting drop columns variables
drop_columns = zero_columns + ['Fwd Header Length.1']

# Number of columns before dropping
col_before = len(cleanDf.columns)

# Dropping the columns
cleanDf = cleanDf.drop(columns=drop_columns)

# Number of columns after dropping
col_after = len(cleanDf.columns)

print(f"Columns Before Dropping: {col_before}")
print(f"Columns After Dropping: {col_after}")
print(f"Columns Dropped: {col_before - col_after}")

Columns Before Dropping: 80
Columns After Dropping: 71
Columns Dropped: 9


**Result:** 9 columns were dropped. The data now has 71 columns.

## Step 3: Outliers and anomalies

### Finding Negative Values
Almost every column in our dataset is a time, a size, or a count, meaning none of these can be negative. 

In [50]:
# Count negative values in every column except for 'Label' or 'source_file'
negative_vals = (cleanDf.drop(columns=['Label', 'source_file']).select_dtypes(exclude=['str','datetime']) < 0).sum()

# Show only the columns that have at least one negative value
negative_vals[negative_vals > 0]


Flow Duration                  107
Flow Bytes/s                    78
Flow Packets/s                 107
Flow IAT Mean                  107
Flow IAT Max                   107
Flow IAT Min                  2880
Fwd IAT Min                     17
Fwd Header Length               35
Bwd Header Length               22
Init_Win_bytes_forward      911012
Init_Win_bytes_backward    1215625
min_seg_size_forward            35
dtype: int64

**Result**

We found negative values in twelve columns, and these columns can be split into three groups. The first group is the negative flow durations and sizes, such as `Flow Duration`, `Fwd Header Length`, and `min_seg_size_forward`. These values are impossible, since a flow cannot last less than zero seconds or send a negative number of bytes. The second group is the negative minimum times between packets, `Flow IAT Min` and `Fwd IAT Min`. These record the shortest gap between two packets in a flow, which cannot be negative. Finally the third group are the negative values in `Init_Win_bytes_forward` and `Init_Win_bytes_backward`, which appear in over a million rows. Since the CICFlowMeter documentation does not explain this value we investigate the values further.



In [51]:
# Rows where the forward window is -1
win_rows = cleanDf[cleanDf['Init_Win_bytes_forward'] == -1]

# Show a few example rows
display(win_rows[['Destination Port', 'ACK Flag Count', 'Init_Win_bytes_forward']].head(100))

,Destination Port,ACK Flag Count,Init_Win_bytes_forward
73,0,0,-1
121,123,0,-1
127,5353,0,-1
133,123,0,-1
134,123,0,-1
...,...,...,...
277,123,0,-1
278,123,0,-1
286,123,0,-1
287,123,0,-1


**Result:** The -1 values appear in 911,012 rows of `Init_Win_bytes_forward` and 1,215,625 rows of `Init_Win_bytes_backward`. The CICFlowMeter documentation does not explain what this value means, and we were not able to find another source that confirms its meaning. However, the value is always exactly -1 rather than a range of random negative numbers, which suggests that it is a code used by CICFlowMeter rather than a measurement error.

**Decision:** We keep these values, since removing them would remove over a million rows based on a value we cannot confirm is an error. Instead, we flag `Init_Win_bytes_forward` and `Init_Win_bytes_backward` as columns whose meaning is uncertain, so they should be interpreted carefully in later analysis.

**Negative Flow Durations and Sizes**

A flow cannot last less than zero seconds, and a header cannot be a negative
number of bytes. We find the rows with negative values in `Flow Duration`, `Fwd Header Length`,
`Bwd Header Length`, or `min_seg_size_forward`, and check which traffic classes they belong to.

In [52]:
# Rows composed of negative values from the four columns we are investigating
broken_rows = ((cleanDf['Flow Duration'] < 0) |
                (cleanDf['Fwd Header Length'] < 0)|
                (cleanDf['Bwd Header Length'] < 0)|
                (cleanDf['min_seg_size_forward'] < 0))

# Count of broken rows
print(f"Rows with negative values: {broken_rows.sum()}")

# Classes associated with these rows
print(cleanDf[broken_rows]['Label'].value_counts())

Rows with negative values: 142
Label
BENIGN    142
Name: count, dtype: int64


**Result:** 142 rows have negative durations or sizes, and all of them are labeled BENIGN.

**Decision:** We remove these 142 rows, since a negative duration or size is impossible and must be an error. Removing them doesn't affect any attack class.

In [53]:
# Count rows before removing 
rows_before = len(cleanDf)

# Remove rows that are broken 
cleanDf = cleanDf.drop(cleanDf[broken_rows].index)

# Count rows after removing
rows_after = len(cleanDf)

print(f"Rows Before Removing: {rows_before}")
print(f"Rows After Removing: {rows_after}")
print(f"Rows Removed: {rows_before - rows_after}")

Rows Before Removing: 2520798
Rows After Removing: 2520656
Rows Removed: 142


**Result:** 142 rows were removed. The data went from 2,520,798 to 2,520,656 rows.

`Flow IAT Min` and `Fwd IAT Min` record the shortest gap between two packets in a flow, which cannot be negative. We find the rows with negative values and check which traffic classes they belong to.

In [54]:
# Rows where the minimum time between packets is negative
neg_time = cleanDf[cleanDf['Flow IAT Min'] < 0]


# Count of rows where the minimum time between packets is negative
print(f"Rows with a negative minimum time: {len(neg_time)}")


# Which negative values appear, and how often
print(neg_time['Flow IAT Min'].value_counts().head())

Rows with a negative minimum time: 2773
Flow IAT Min
-1     2642
-12      30
-2       25
-3       17
-4       16
Name: count, dtype: int64


In [55]:
# Rows with a negative minimum time, per class
neg_per_label = neg_time['Label'].value_counts()

# All rows per class
total = cleanDf['Label'].value_counts()

# Percentage of each class that has a negative minimum time
share = neg_per_label / total * 100

# Put the three columns side by side
summary = pd.DataFrame({"negative_time": neg_per_label, "total": total, "share_%": share.round(2)})
display(summary.fillna(0))

,negative_time,total,share_%
Label,,,
BENIGN,2580.0,2094915,0.12
Bot,0.0,1948,0.00
DDoS,19.0,128014,0.01
DoS GoldenEye,5.0,10286,0.05
DoS Hulk,158.0,172846,0.09
DoS Slowhttptest,0.0,5228,0.00
DoS slowloris,0.0,5385,0.00
FTP-Patator,4.0,5931,0.07
Heartbleed,4.0,11,36.36


**Result:** 2,773 rows have a negative minimum time between packets, and the most common value is -1, which means it is only one microsecond below zero. These rows include 4 of the 11 Heartbleed flows (36%).

**Decision:** Instead of removing these rows, we change the negative value to 0, which is the closest possible real value. We do this since removing the rows would cause more than a third of the Heartbleed values to be removed.

In [56]:
# Set negative minimum times to 0
cleanDf.loc[cleanDf['Flow IAT Min'] < 0, 'Flow IAT Min'] = 0
cleanDf.loc[cleanDf['Fwd IAT Min'] < 0, 'Fwd IAT Min'] = 0

# Check that no negative values are left
print(f"Negative Flow IAT Min left: {(cleanDf['Flow IAT Min'] < 0).sum()}")
print(f"Negative Fwd IAT Min left: {(cleanDf['Fwd IAT Min'] < 0).sum()}")
print(f"Rows: {len(cleanDf)}")

Negative Flow IAT Min left: 0
Negative Fwd IAT Min left: 0
Rows: 2520656


**Result:** All Negative minimum times were changed to 0. No rows were removed, so the data still has 2,520,656 rows.

### Extreme Values: Very Large Packets


According to Cloudflare 1,500 bytes is the maximum MTU size, which should be the largest packet size for most networks.

source: (https://www.cloudflare.com/learning/network-layer/what-is-mtu/)

In [57]:
# Flows with a packet larger than 1,500 bytes
large_rows = cleanDf[cleanDf['Max Packet Length'] > 1500]

# Counting large packet flows by label 
big_per_label = large_rows['Label'].value_counts()

# Counting all flows per label
total = cleanDf['Label'].value_counts()

# Percentage of each label that has a large packet
share = big_per_label / total * 100

# Putting the three columns side by side
summary = pd.DataFrame({"large_packets": big_per_label,
                        "total": total,
                        "share_%": share.round(1)})

# Display the table (labels with no large packets show 0)
display(summary.fillna(0))

,large_packets,total,share_%
Label,,,
BENIGN,221962.0,2094915,10.6
Bot,52.0,1948,2.7
DDoS,81346.0,128014,63.5
DoS GoldenEye,7356.0,10286,71.5
DoS Hulk,154852.0,172846,89.6
DoS Slowhttptest,264.0,5228,5.0
DoS slowloris,4.0,5385,0.1
FTP-Patator,0.0,5931,0.0
Heartbleed,11.0,11,100.0


**Result:** Large packets appear much more often in the DoS and DDoS attacks than in benign traffic. 89.6% of DoS Hulk, 71.5% of DoS GoldenEye, and 63.5% of DDoS flows have a packet larger than 1,500 bytes, compared to only 10.6% of BENIGN flows.

**Decision:** We keep these values. Although they are larger than the usual maximum packet size, they appear in too many flows to be random errors. The reasoning for keeping them is that these extreme values are part of what makes these attacks different from benign traffic, so removing them would remove the patterns our project is trying to study.

## Step 4: Data Types and Parsing

In [58]:
# Count how many columns have each data type
print(cleanDf.dtypes.value_counts())

# Show the columns that are text (not numbers)
print(cleanDf.select_dtypes(exclude='number').columns.tolist())

int64      45
float64    24
str         2
Name: count, dtype: int64
['Label', 'source_file']


**Result:** All 69 feature columns were read as numbers (45 whole numbers and 24 decimals), and the only text columns are `Label` and `source_file`. No column needed to be converted.

Our dataset does not contain any date or time columns, so no date parsing was needed. It also does not contain identifiers such as IP addresses or flow IDs.

## Step 5: Transformations

### Setting the Target
Our first and fourth research questions compare benign traffic to all attacks, so we create a binary column, `is_attack`, from the `Label` column: 0 for BENIGN and 1 for any attack.

In [59]:
# Create is_attack: 0 if the label is BENIGN, 1 if it is any attack
cleanDf['is_attack'] = (cleanDf['Label'] != 'BENIGN').astype(int)

# Count each value
print(cleanDf['is_attack'].value_counts())

# Percentage of flows that are attacks
print(f"Percent attacks: {round(cleanDf['is_attack'].mean() * 100, 2)}%")

is_attack
0    2094915
1     425741
Name: count, dtype: int64
Percent attacks: 16.89%


**Result:** 2,094,915 flows are benign (0) and 425,741 are attacks (1), so 16.89% of the cleaned data are attacks. The `Label` column is kept as well, since our other research questions compare the individual attack types.

**Transformations Not applied:** Scaling and normalization are not applied here, since they will be applied during analysis using only the training data, to avoid data leakage. No other transformations (log transforms, aggregation, or further dimensionality reduction) were needed for cleaning.

## Step 6: Saving the Cleaned Data
We save the cleaned data to `data/processed/`, so it can be used for analysis without rerunning every cleaning step. The raw files in `data/raw/` were never changed, so running this notebook from top to bottom recreates the cleaned data from the original files.

In [60]:
# Save the cleaned data to the processed folder
cleanDf.to_csv("../data/processed/cicids2017_clean.csv", index=False)

# Final size of the cleaned data
print(f"Saved {len(cleanDf)} rows and {len(cleanDf.columns)} columns")

Saved 2520656 rows and 72 columns
